In [ ]:
from multi_llm_debate.analysis.correct_rate_by_round import (
    calculate_correct_rate_by_round,
    calculate_majority_vote_correct_rate
)
from multi_llm_debate.analysis.classify_task_difficulty import (
    analyze_task_difficulty,
)
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Data path
data_path = Path("../output/bool_q/processed_data.csv")

# Load dataset
dataframe = pd.read_csv(data_path)

# Get all model directories from the specified path
model_dirs = Path("../data/bool_q").glob('*')

for model_dir in model_dirs:
    if model_dir.is_dir():
        print(f"Processing model directory: {model_dir}")
        
        # Analyze task difficulty
        result_df = analyze_task_difficulty(
            model_dir=model_dir,
            dataframe=dataframe,
            accuracy_threshold=0.4,
        )

        easy_df = result_df[result_df["difficulty"] == 0]
        hard_df = result_df[result_df["difficulty"] == 1]
        
        overall_correct_rate_df = calculate_correct_rate_by_round(
            dataframe=result_df,
            model_dir=model_dir,
            max_round_number=10
        )
        overall_majority_vote_rate = calculate_majority_vote_correct_rate(
            dataframe=dataframe,
            model_dir=model_dir,
        )
        # Calculate correct rate by round for easy and hard tasks
        easy_correct_rate_by_round_df = calculate_correct_rate_by_round(
            dataframe=easy_df,
            model_dir=model_dir,
            max_round_number=10
        )
        easy_majority_vote_rate = calculate_majority_vote_correct_rate(
            dataframe=easy_df,
            model_dir=model_dir,
        )
        hard_correct_rate_by_round_df = calculate_correct_rate_by_round(
            dataframe=hard_df,
            model_dir=model_dir,
            max_round_number=10
        )
        hard_majority_vote_rate = calculate_majority_vote_correct_rate(
            dataframe=hard_df,
            model_dir=model_dir,
        )
        # print(hard_correct_rate_by_round_df.head())
        # Get numeric columns (rounds 0-10)
        easy_rates = easy_correct_rate_by_round_df[
            easy_correct_rate_by_round_df['metric'] == 'majority'
        ].iloc[0, 2:].values
        hard_rates = hard_correct_rate_by_round_df[
            hard_correct_rate_by_round_df['metric'] == 'majority'
        ].iloc[0, 2:].values
        overall_rates = overall_correct_rate_df[
            overall_correct_rate_df['metric'] == 'majority'
        ].iloc[0, 2:].values
        rounds = range(len(easy_rates))

        # Create figure and axis for plotting
        plt.figure(figsize=(10, 6))

        # Plot both lines
        plt.plot(rounds, easy_rates, 'b-o', label='Easy Tasks', linewidth=2)
        plt.plot(rounds, hard_rates, 'r-s', label='Hard Tasks', linewidth=2)
        plt.plot(rounds, overall_rates, 'g-d', label='Overall', linewidth=2)

        # Add horizontal lines for majority vote rates
        plt.axhline(y=easy_majority_vote_rate, color='blue', linestyle='--', label='Easy Majority Vote Rate')
        plt.axhline(y=hard_majority_vote_rate, color='red', linestyle='--', label='Hard Majority Vote Rate')
        plt.axhline(y=overall_majority_vote_rate, color='green', linestyle='--', label='Overall Majority Vote Rate')
        
        # Customize plot
        plt.title(f'Correct Rate by Round for Model {model_dir.name} (Task Difficulty)', pad=15)
        plt.xlabel('Round Number')
        plt.ylabel('Correct Rate')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.legend()

        # Set y-axis limits and ticks
        plt.ylim(0, 1)
        plt.yticks(np.arange(0, 1.1, 0.1))

        # Set x-axis ticks
        plt.xticks(rounds)

        # Show plot
        plt.tight_layout()
        plt.show()


Processing model directory: ../data/bool_q/llama2(3)+llama3(3)
Error processing task directory ../data/bool_q/llama2(3)+llama3(3)/4780: Answer not recognized
Error processing task directory ../data/bool_q/llama2(3)+llama3(3)/7627: Answer not recognized
